# Public release note

This is a sanitized public copy of `omniASR_20_K4_KenLM_V6_acoustic_WER(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# 20.K4 — KenLM V6 : tuning WER acoustique sans fuite

Ce notebook compare les finalistes construits en **20.K3** à la configuration exacte du score leaderboard **0.36878**. Il utilise le même modèle acoustique (`step_1250` BF16) et active uniquement l'adaptateur LoRA **Maasai**.

Garde-fous : choix sur `tune` uniquement, audit apparié une seule fois, `batch_size=1`, aucun audio test lu, aucune configuration active écrasée et aucune soumission créée. Un candidat accepté est seulement publié pour le futur audit edge **20.K5**.

Pré-requis : 20.K2 et 20.K3 terminés avec PASS, package edge 19.7f-c présent sur Drive, GPU L4 ou A100 et RAM système augmentée. Durée typique : **55 min à 2 h 15** ; le décodage KenLM est surtout limité par le CPU.


## Ordre d'exécution

1. Exécuter **20.K4.A**. Si la cellule demande un redémarrage, redémarrer le runtime puis rejouer 20.K4.A.
2. Exécuter **20.K4.B** jusqu'au statut final. La cellule reprend automatiquement les groupes déjà terminés.
3. Exécuter **20.K4.C** pour lire le rapport. Ne pas lancer 17.2 à ce stade.


In [ ]:
# 20.K4.A — Setup Colab idempotent (nouvel environnement)
from google.colab import drive
drive.mount('/content/drive')

import importlib.metadata as md
import os, subprocess, sys

loaded_before = {name for name in ("torch", "numpy") if name in sys.modules}
def version_or_none(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

before = {name: version_or_none(name) for name in ("torch", "numpy")}
repo = "/content/omnilingual-asr"
if not os.path.isdir(os.path.join(repo, ".git")):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/facebookresearch/omnilingual-asr.git", repo,
    ], check=True)

requirements = [
    repo,
    "pyarrow>=20.0.0",
    "pandas>=2.2.0",
    "numpy<3",
    "soundfile>=0.12.1",
    "jiwer>=3.0.0",
    "pyctcdecode>=0.5.0",
    "kenlm>=0.3.0",
    "ftfy>=6.2.0",
    "regex>=2024.0.0",
    "rapidfuzz>=3.6.0",
    "psutil>=5.9.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)

# fairseq2 peut imposer une autre version de torch. Aligner les extensions Colab
# évite les erreurs de symboles binaires au premier import.
torch_base = md.version("torch").split("+")[0]
torch_mm = ".".join(torch_base.split(".")[:2])
tv_for_torch = {
    "2.8": "0.23.0", "2.9": "0.24.0", "2.10": "0.25.0", "2.11": "0.26.0"
}
targets = {"torchaudio": torch_base, "torchvision": tv_for_torch.get(torch_mm)}
realigned = False
for package, target in targets.items():
    if not target:
        continue
    current = version_or_none(package)
    if current and current.split("+")[0] != target:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"{package}=={target}"],
            check=True,
        )
        realigned = True

after = {name: version_or_none(name) for name in ("torch", "numpy")}
changed_loaded = [
    name for name in loaded_before
    if before[name] and after[name] and before[name].split("+")[0] != after[name].split("+")[0]
]
if changed_loaded or realigned:
    print("⚠️ Des bibliothèques binaires ont été réalignées.")
    print("Runtime > Redémarrer la session, puis réexécuter uniquement 20.K4.A et 20.K4.B.")
    raise SystemExit("REDÉMARRAGE COLAB REQUIS")

import torch
assert torch.cuda.is_available(), "Activer un GPU L4/A100 dans Runtime > Modifier le type d'exécution."
assert torch.cuda.is_bf16_supported(), "Le GPU sélectionné ne prend pas BF16 en charge."
commit = subprocess.run(
    ["git", "-C", repo, "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("✅ Setup 20.K4 prêt")
print("GPU :", torch.cuda.get_device_name(0), "| VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "Gio")
print("torch :", md.version("torch"), "| fairseq2 :", md.version("fairseq2"))
print("omnilingual-asr commit :", commit)


## 20.K4.B — Préflight, extraction acoustique, grille TUNE, audit et publication candidate

Cette cellule est volontairement unique et reprenable : après une coupure, la relancer telle quelle. Les logits restent en RAM et ne sont pas copiés sur Drive.


In [ ]:
"""20.K4 - tuning WER acoustique des KenLM V6 sans fuite.

Ce script est destine a etre execute dans une seule cellule Colab APRES la
cellule de setup du notebook 20.K4. Il est autonome vis-a-vis des anciens
objets Python : tous les artefacts utiles sont relus et verifies sur Drive.

Regles importantes :
- runtime acoustique exact du score 0.36878 : base step_1250 BF16 + LoRA mas ;
- batch acoustique strictement egal a 1 ;
- choix des candidats sur TUNE uniquement ; AUDIT ne sert qu'une fois ;
- aucun ecrasement de la configuration active et aucune inference test ;
- reprise atomique groupe par groupe, sans conserver les logits sur Drive.
"""

from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import ftfy
import jiwer
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import regex as uregex
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from ftfy.badness import badness as ftfy_badness
from pyctcdecode import build_ctcdecoder
from rapidfuzz.fuzz import ratio as edit_ratio


# ---------------------------------------------------------------------------
# 1. Constantes figees
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/afrivoices_project")
FT_ROOT = PROJECT_ROOT / "finetune_runs/experiment_run"
REPORT_ROOT = FT_ROOT / "reports"
K2_REPORT_ROOT = REPORT_ROOT / "kenlm_v6_20_K2"
K3_REPORT_ROOT = REPORT_ROOT / "kenlm_v6_20_K3"
K4_REPORT_ROOT = REPORT_ROOT / "kenlm_v6_20_K4"

EXPECTED_K2_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"
EXPECTED_K2_CONTRACT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K3_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"
EXPECTED_K3_CONTRACT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)

LANGUAGES = ("sw", "kik", "kln", "luo", "som", "mas")
ACTIVE_GROUPS = tuple(
    ["sw|unscripted"]
    + [
        f"{language}|{domain}"
        for language in LANGUAGES[1:]
        for domain in ("scripted", "unscripted")
    ]
)
ALL_RUNTIME_GROUPS = tuple(
    f"{language}|{domain}"
    for language in LANGUAGES
    for domain in ("scripted", "unscripted")
)
LANG_TO_OMNI = {
    "sw": "swh_Latn",
    "kik": "kik_Latn",
    "kln": "kln_Latn",
    "luo": "luo_Latn",
    "som": "som_Latn",
    "mas": "saq_Latn",
}

BASE_PARAMETERS = 975_675_056
MAS_ADAPTER_PARAMETERS = 9_898_496
TOTAL_NEURAL_PARAMETERS = 985_573_552
assert BASE_PARAMETERS + MAS_ADAPTER_PARAMETERS == TOTAL_NEURAL_PARAMETERS
assert TOTAL_NEURAL_PARAMETERS < 1_000_000_000

TUNE_MAX_CLIPS = 64
AUDIT_MIN_CLIPS = 24
AUDIT_MAX_CLIPS = 120
BOOTSTRAP_REPETITIONS = 2_000
BOOTSTRAP_SEED = 20_400_006
BATCH_SIZE = 1

COARSE_ALPHAS = (0.0, 0.4, 0.8, 1.2)
COARSE_BETAS = (0.0, 1.0, 2.0)
FINE_ALPHA_OFFSETS = (-0.2, 0.0, 0.2)
FINE_BETA_OFFSETS = (-0.5, 0.0, 0.5)
BEAM_WIDTHS_FINAL = (100, 250)

MIN_AUDIT_ABSOLUTE_GAIN = 0.001
MIN_PROBABILITY_BETTER = 0.90
MAX_DURATION_STRATUM_REGRESSION = 0.005
MIN_STRATUM_REFERENCE_WORDS = 1_000

LOCAL_AUDIO_ROOT = Path("/content/kenlm_k4_dev_audio")
LOCAL_ASSET_ROOT = Path("/content/kenlm_k4_assets")
LOCAL_CARD_NAME = "afrivoices_step1250_edge_bf16_k4"

assert torch.cuda.is_available(), "20.K4 requiert un GPU ; un L4 ou A100 suffit."
assert torch.cuda.is_bf16_supported(), "Le GPU doit supporter BF16."
assert Path("/content/persistent_storage").is_dir(), "Monter Google Drive avant 20.K4."


# ---------------------------------------------------------------------------
# 2. Utilitaires immuables et ecritures atomiques
# ---------------------------------------------------------------------------

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path | str, block_size: int = 16 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str,
    )


def canonical_sha256(value: Any) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def read_json(path: Path | str) -> dict[str, Any]:
    with open(path, encoding="utf-8") as handle:
        value = json.load(handle)
    assert isinstance(value, dict), path
    return value


def atomic_json(value: Any, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_csv(frame: pd.DataFrame, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)
    print("csv  ->", path)


def atomic_parquet(frame: pd.DataFrame, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_parquet(temporary, index=False)
    os.replace(temporary, path)
    print("parquet ->", path)


def artifact_metadata(path: Path | str) -> dict[str, Any]:
    path = Path(path)
    return {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


def require_child(path: Path | str, root: Path | str) -> Path:
    path = Path(path).resolve()
    root = Path(root).resolve()
    assert path == root or root in path.parents, (path, root)
    return path


def validate_artifacts(root: Path, artifacts: dict[str, Any]) -> None:
    for relative, metadata in artifacts.items():
        path = require_child(root / relative, root)
        assert path.is_file(), path
        assert path.stat().st_size == int(metadata["bytes"]), (path, "size")
        assert sha256_file(path) == metadata["sha256"], (path, "sha256")


def copy_verified(source: Path, expected_sha256: str, label: str) -> Path:
    target_dir = LOCAL_ASSET_ROOT / expected_sha256[:16]
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / f"{label}{source.suffix}"
    if target.is_file() and sha256_file(target) == expected_sha256:
        return target
    temporary = target.with_name(target.name + f".tmp-{os.getpid()}")
    shutil.copy2(source, temporary)
    assert sha256_file(temporary) == expected_sha256
    os.replace(temporary, target)
    return target


def package_versions() -> dict[str, str]:
    output = {}
    for package in (
        "torch", "fairseq2", "omnilingual-asr", "pyarrow", "pandas",
        "numpy", "soundfile", "jiwer", "pyctcdecode", "kenlm", "ftfy",
        "regex", "rapidfuzz",
    ):
        try:
            output[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            output[package] = "UNKNOWN"
    return output


# ---------------------------------------------------------------------------
# 3. Validation stricte de K2/K3 et du runtime 0.36878
# ---------------------------------------------------------------------------

K2_LATEST = read_json(K2_REPORT_ROOT / "LATEST_PASS.json")
assert K2_LATEST["cell"] == "20.K2"
assert K2_LATEST["status"] == "PASS_READY_FOR_20_K3_KENLM_BUILD"
assert K2_LATEST["run_id"] == EXPECTED_K2_RUN_ID
assert K2_LATEST["contract_sha256"] == EXPECTED_K2_CONTRACT_SHA256
assert int(K2_LATEST.get("audio_bytes_read", 0)) == 0
K2_ROOT = require_child(K2_LATEST["corpus_root"], PROJECT_ROOT)
K2_COMPLETE_PATH = require_child(K2_LATEST["complete"], K2_ROOT)
assert sha256_file(K2_COMPLETE_PATH) == K2_LATEST["complete_sha256"]
K2_COMPLETE = read_json(K2_COMPLETE_PATH)
assert K2_COMPLETE["status"] == "PASS_READY_FOR_20_K3_KENLM_BUILD"
assert K2_COMPLETE["run_id"] == EXPECTED_K2_RUN_ID
assert K2_COMPLETE["contract_sha256"] == EXPECTED_K2_CONTRACT_SHA256
validate_artifacts(K2_ROOT, K2_COMPLETE["artifacts"])

K3_LATEST = read_json(K3_REPORT_ROOT / "LATEST_PASS.json")
assert K3_LATEST["cell"] == "20.K3"
assert K3_LATEST["status"] == "PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING"
assert K3_LATEST["run_id"] == EXPECTED_K3_RUN_ID
assert K3_LATEST["contract_sha256"] == EXPECTED_K3_CONTRACT_SHA256
K3_ROOT = require_child(K3_LATEST["model_root"], FT_ROOT)
K3_COMPLETE_PATH = require_child(K3_LATEST["complete"], K3_ROOT)
assert sha256_file(K3_COMPLETE_PATH) == K3_LATEST["complete_sha256"]
K3_COMPLETE = read_json(K3_COMPLETE_PATH)
assert K3_COMPLETE["status"] == "PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING"
assert K3_COMPLETE["run_id"] == EXPECTED_K3_RUN_ID
assert K3_COMPLETE["contract_sha256"] == EXPECTED_K3_CONTRACT_SHA256
validate_artifacts(K3_ROOT, K3_COMPLETE["artifacts"])
K3_SELECTION = read_json(K3_ROOT / "selection.json")
assert set(K3_SELECTION) == set(LANGUAGES)
for language in LANGUAGES:
    finalists = K3_SELECTION[language]["v6_finalists"]
    assert 1 <= len(finalists) <= 2
    for finalist in finalists:
        relative = finalist["binary_relative_path"]
        assert relative in K3_COMPLETE["artifacts"]
        assert K3_COMPLETE["artifacts"][relative]["sha256"] == (
            finalist["binary_sha256"]
        )


EDGE_AUDIT_PATH = REPORT_ROOT / "lora_edge_bf16_audit_v2.json"
EDGE_AUDIT = read_json(EDGE_AUDIT_PATH)
assert EDGE_AUDIT.get("status") == "PASS_READY_FOR_TEST_INFERENCE", EDGE_AUDIT.get("status")


def parameter_proofs(value: Any) -> list[int]:
    output: list[int] = []
    if isinstance(value, dict):
        for key, item in value.items():
            if "parameter" in str(key).lower() and isinstance(item, (int, float)):
                output.append(int(item))
            output.extend(parameter_proofs(item))
    elif isinstance(value, list):
        for item in value:
            output.extend(parameter_proofs(item))
    return output


assert TOTAL_NEURAL_PARAMETERS in parameter_proofs(EDGE_AUDIT), (
    "Preuve du nombre de parametres absente du rapport edge",
    sorted(set(parameter_proofs(EDGE_AUDIT))),
)

# Ce package est fige : ne jamais choisir silencieusement le dossier le plus recent.
EDGE_PACKAGE = require_child(
    FT_ROOT / "final_lora_edge_candidate_v1/REPLACE_WITH_LOCAL_RUN_ID", FT_ROOT
)
assert EDGE_PACKAGE.is_dir(), EDGE_PACKAGE
for key in ("package", "package_dir", "edge_package"):
    if EDGE_AUDIT.get(key):
        assert os.path.realpath(EDGE_AUDIT[key]) == os.path.realpath(EDGE_PACKAGE)
EDGE_MANIFEST_PATH = EDGE_PACKAGE / "edge_manifest_v2.json"
assert EDGE_MANIFEST_PATH.is_file(), EDGE_MANIFEST_PATH
EDGE_MANIFEST = read_json(EDGE_MANIFEST_PATH)
BASE_WEIGHT = EDGE_PACKAGE / "base_step1250_bf16.pt"
MAS_ADAPTER = EDGE_PACKAGE / "adapter_mas_step1250.pt"
BASELINE_CONFIG_PATH = EDGE_PACKAGE / "kenlm_tuning_by_domain.json"
BASE_WEIGHT_SHA256 = sha256_file(BASE_WEIGHT)
MAS_ADAPTER_SHA256 = sha256_file(MAS_ADAPTER)
BASELINE_CONFIG_SHA256 = sha256_file(BASELINE_CONFIG_PATH)


def manifest_sha_proofs(value: Any, basename: str) -> set[str]:
    """Recupere les SHA lies a un asset, quelle que soit la mise en page JSON."""
    proofs: set[str] = set()
    if isinstance(value, dict):
        serialized_strings = [str(key) for key in value]
        serialized_strings.extend(
            str(item) for item in value.values() if isinstance(item, str)
        )
        mentions_asset = any(basename in item for item in serialized_strings)
        if mentions_asset:
            for key, item in value.items():
                if "sha256" in str(key).lower() and isinstance(item, str):
                    if re.fullmatch(r"[0-9a-fA-F]{64}", item):
                        proofs.add(item.lower())
        for key, item in value.items():
            if str(key) == basename and isinstance(item, dict):
                for child_key, child_value in item.items():
                    if (
                        "sha256" in str(child_key).lower()
                        and isinstance(child_value, str)
                        and re.fullmatch(r"[0-9a-fA-F]{64}", child_value)
                    ):
                        proofs.add(child_value.lower())
                proofs.update(manifest_sha_proofs(item, basename))
            else:
                proofs.update(manifest_sha_proofs(item, basename))
    elif isinstance(value, list):
        for item in value:
            proofs.update(manifest_sha_proofs(item, basename))
    return proofs


def all_manifest_sha256(value: Any) -> set[str]:
    proofs: set[str] = set()
    if isinstance(value, dict):
        for key, item in value.items():
            if (
                "sha256" in str(key).lower()
                and isinstance(item, str)
                and re.fullmatch(r"[0-9a-fA-F]{64}", item)
            ):
                proofs.add(item.lower())
            proofs.update(all_manifest_sha256(item))
    elif isinstance(value, list):
        for item in value:
            proofs.update(all_manifest_sha256(item))
    return proofs


for asset_path, actual_sha in (
    (BASE_WEIGHT, BASE_WEIGHT_SHA256),
    (MAS_ADAPTER, MAS_ADAPTER_SHA256),
    (BASELINE_CONFIG_PATH, BASELINE_CONFIG_SHA256),
):
    proofs = manifest_sha_proofs(EDGE_MANIFEST, asset_path.name)
    if not proofs:
        # Certaines versions du manifest nomment les roles (base/adapter/config)
        # sans repeter le basename ; le SHA doit neanmoins y etre present.
        proofs = all_manifest_sha256(EDGE_MANIFEST)
    assert actual_sha in proofs, (asset_path.name, actual_sha, proofs)

BASELINE_CONFIG = read_json(BASELINE_CONFIG_PATH)
assert set(BASELINE_CONFIG) == set(ALL_RUNTIME_GROUPS), sorted(BASELINE_CONFIG)
for key, config in BASELINE_CONFIG.items():
    assert Path(config["lm_bin"]).is_file(), (key, config["lm_bin"])
    assert float(config["alpha"]) >= 0
    assert int(config["beam"]) in (100, 250)

PLAN_PATH = (
    REPORT_ROOT
    / "lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID/lora_plan.parquet"
)
DEV_SELECTION_PATH = REPORT_ROOT / "balanced_v4/selected_dev_v4.parquet"
TRAIN_SELECTION_PATH = REPORT_ROOT / "balanced_v4/selected_records_v4.parquet"
for required in (PLAN_PATH, DEV_SELECTION_PATH, TRAIN_SELECTION_PATH):
    assert required.is_file(), required
PLAN_SHA256 = sha256_file(PLAN_PATH)
DEV_SELECTION_SHA256 = sha256_file(DEV_SELECTION_PATH)
TRAIN_SELECTION_SHA256 = sha256_file(TRAIN_SELECTION_PATH)

print("=== PREFLIGHT 20.K4 ===")
print("K2 :", EXPECTED_K2_RUN_ID)
print("K3 :", EXPECTED_K3_RUN_ID)
print("Package acoustique :", EDGE_PACKAGE)
print("Base BF16 :", BASE_WEIGHT_SHA256[:16])
print("LoRA mas  :", MAS_ADAPTER_SHA256[:16])
print("Parametres neuraux :", f"{TOTAL_NEURAL_PARAMETERS:,}")


# ---------------------------------------------------------------------------
# 4. Normalisation identique a K2/K3 et preparation du dev
# ---------------------------------------------------------------------------

MOJIBAKE_RE = re.compile(
    r"(?:Ã.|Â.|Ä.|Å.|â[€‚„…†‡ˆ‰Š‹ŒŽ‘’“”•–—˜™š›œžŸ]|ï¿½)"
)
APOSTROPHE_DASH_TRANSLATION = str.maketrans(
    {
        "’": "'", "‘": "'", "ʼ": "'", "`": "'",
        "–": " ", "—": " ", "‐": " ", "‑": " ",
    }
)


def suspect_score(text: str) -> int:
    return int(ftfy_badness(text)) + 4 * text.count("\ufffd") + len(
        MOJIBAKE_RE.findall(text)
    )


def letter_ratio(text: str) -> float:
    letters = sum(
        char.isalpha() or unicodedata.category(char).startswith("M")
        for char in text
    )
    return letters / max(1, len(text))


def repair_text(raw: Any) -> str | None:
    if raw is None:
        return None
    text = unicodedata.normalize("NFC", str(raw))
    if "\ufffd" in text:
        return None
    if not MOJIBAKE_RE.search(text):
        return text
    candidate = unicodedata.normalize("NFC", ftfy.fix_text(text, normalization="NFC"))
    safe = (
        "\ufffd" not in candidate
        and suspect_score(candidate) < suspect_score(text)
        and letter_ratio(candidate) + 0.02 >= letter_ratio(text)
        and 0.8 <= len(candidate) / max(1, len(text)) <= 1.2
    )
    return candidate if safe else text


def normalize_text(raw: Any) -> str:
    repaired = repair_text(raw)
    assert repaired is not None
    text = unicodedata.normalize(
        "NFC", repaired.translate(APOSTROPHE_DASH_TRANSLATION).casefold()
    )
    text = uregex.sub(r"[^\p{L}\p{M}\p{N}'-]+", " ", text)
    text = uregex.sub(
        r"(?<![\p{L}\p{M}])['-]|['-](?![\p{L}\p{M}])", " ", text
    )
    text = unicodedata.normalize("NFC", " ".join(text.split()))
    assert text and letter_ratio(text) >= 0.45, raw
    return text


def normalize_hypothesis(raw: Any) -> str:
    """Normalisation de sortie CTC ; une hypothese vide est parfaitement valide."""
    repaired = repair_text(raw)
    if repaired is None:
        return ""
    text = unicodedata.normalize(
        "NFC", repaired.translate(APOSTROPHE_DASH_TRANSLATION).casefold()
    )
    text = uregex.sub(r"[^\p{L}\p{M}\p{N}'-]+", " ", text)
    text = uregex.sub(
        r"(?<![\p{L}\p{M}])['-]|['-](?![\p{L}\p{M}])", " ", text
    )
    return unicodedata.normalize("NFC", " ".join(text.split()))


# Contrat WER exact utilise par 18.8/19.7 et par la soumission 0.36878.
# Il est volontairement distinct du normaliseur de corpus KenLM ci-dessus.
_V4_MOJIBAKE_MARKERS = set("ÃÂÅâðÐÑ¤¦¨©¬®¯°±²³´µ¶·¸¹º»¼½¾¿")
_V4_PUNCT_RE = re.compile(r"[^\w\s'’\-]", re.UNICODE)
_V4_WS_RE = re.compile(r"\s+")


def v4_text_badness(value: Any) -> int:
    value = str(value)
    score = 50 * value.count("�")
    score += 6 * sum(char in _V4_MOJIBAKE_MARKERS for char in value)
    score += 10 * sum(
        ord(char) < 32 and char not in "\n\t\r" for char in value
    )
    return score


def v4_repair_mojibake(value: Any) -> str:
    raw = str(value or "")
    candidates = [(raw, "none")]
    if any(char in _V4_MOJIBAKE_MARKERS for char in raw) or "�" in raw:
        for encoding in ("latin-1", "cp1252"):
            try:
                candidates.append((raw.encode(encoding).decode("utf-8"), encoding))
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
    best, _ = min(candidates, key=lambda item: v4_text_badness(item[0]))
    return best if v4_text_badness(best) < v4_text_badness(raw) else raw


def normalize_wer_text(value: Any) -> str:
    value = unicodedata.normalize("NFC", v4_repair_mojibake(value)).lower()
    value = _V4_PUNCT_RE.sub(" ", value).replace("_", " ")
    return _V4_WS_RE.sub(" ", value).strip()


def chunk_k2_text(text: str, maximum_words: int = 80) -> list[str]:
    words = text.split()
    if len(words) <= maximum_words:
        return [text]
    chunks = [
        words[index : index + maximum_words]
        for index in range(0, len(words), maximum_words)
    ]
    if len(chunks) > 1 and len(chunks[-1]) < 8:
        needed = min(8 - len(chunks[-1]), len(chunks[-2]) - 1)
        if needed > 0:
            chunks[-1] = chunks[-2][-needed:] + chunks[-1]
            chunks[-2] = chunks[-2][:-needed]
    return [" ".join(chunk) for chunk in chunks if chunk]


# Index near-duplicate identique a K2 (mode heldout).
MIN_NEAR_WORDS = 12
MIN_NEAR_CHARS = 80
MINHASH_COMPONENTS = 16
MINHASH_BANDS = 8
HELDOUT_EDIT = 0.98
HELDOUT_JACCARD = 0.94
HASH_PERSON = b"K2SHING"


def k2_hash64(payload: bytes) -> int:
    return int.from_bytes(
        hashlib.blake2b(payload, digest_size=8, person=HASH_PERSON).digest(),
        "little",
    )


MINHASH_SEEDS = tuple(
    k2_hash64(f"minhash:{index}".encode("ascii"))
    for index in range(MINHASH_COMPONENTS)
)
MINHASH_SEEDS_ARRAY = np.asarray(MINHASH_SEEDS, dtype=np.uint64)


def k2_shingles(text: str) -> tuple[int, ...]:
    words = text.split()
    grams = (
        ("\x1f".join(words[index : index + 3]) for index in range(len(words) - 2))
        if len(words) >= 3
        else ("\x1f".join(words),)
    )
    values = sorted({k2_hash64(gram.encode("utf-8")) for gram in grams if gram})
    assert values
    return tuple(values)


def k2_signature(shingles: tuple[int, ...]) -> tuple[int, ...]:
    values = np.asarray(shingles, dtype=np.uint64)[:, None]
    mixed = np.bitwise_xor(values, MINHASH_SEEDS_ARRAY[None, :])
    with np.errstate(over="ignore"):
        mixed = mixed + np.uint64(0x9E3779B97F4A7C15)
        mixed = (
            (mixed ^ (mixed >> np.uint64(30)))
            * np.uint64(0xBF58476D1CE4E5B9)
        )
        mixed = (
            (mixed ^ (mixed >> np.uint64(27)))
            * np.uint64(0x94D049BB133111EB)
        )
        mixed = mixed ^ (mixed >> np.uint64(31))
    return tuple(int(value) for value in mixed.min(axis=0))


def k2_band_keys(signature: tuple[int, ...]) -> tuple[tuple[int, ...], ...]:
    width = MINHASH_COMPONENTS // MINHASH_BANDS
    return tuple(
        (band, *signature[band * width : (band + 1) * width])
        for band in range(MINHASH_BANDS)
    )


def k2_near_features(
    text: str,
) -> tuple[int, tuple[str, ...], tuple[int, ...], tuple[int, ...]] | None:
    word_count = len(text.split())
    if word_count < MIN_NEAR_WORDS and len(text) < MIN_NEAR_CHARS:
        return None
    shingles = k2_shingles(text)
    return (
        word_count,
        tuple(uregex.findall(r"\p{N}+", text)),
        shingles,
        k2_signature(shingles),
    )


class K2NearIndex:
    def __init__(self) -> None:
        self.texts: list[str] = []
        self.word_counts: list[int] = []
        self.numbers: list[tuple[str, ...]] = []
        self.shingles: list[tuple[int, ...]] = []
        self.buckets: dict[tuple[int, ...], list[int]] = defaultdict(list)

    def add(self, text: str) -> None:
        features = k2_near_features(text)
        if features is None:
            return
        word_count, numbers, shingles, signature = features
        index = len(self.texts)
        self.texts.append(text)
        self.word_counts.append(word_count)
        self.numbers.append(numbers)
        self.shingles.append(shingles)
        for key in k2_band_keys(signature):
            self.buckets[key].append(index)

    def find(self, text: str) -> int | None:
        features = k2_near_features(text)
        if features is None:
            return None
        word_count, numbers, shingles_tuple, signature = features
        shingles = set(shingles_tuple)
        candidates: set[int] = set()
        for key in k2_band_keys(signature):
            candidates.update(self.buckets.get(key, ()))
        for index in sorted(candidates):
            other_word_count = self.word_counts[index]
            length_ratio = min(word_count, other_word_count) / max(
                1, word_count, other_word_count
            )
            if length_ratio < 0.90 or numbers != self.numbers[index]:
                continue
            other_shingles = set(self.shingles[index])
            union = shingles | other_shingles
            jaccard = len(shingles & other_shingles) / len(union) if union else 1.0
            if jaccard < HELDOUT_JACCARD:
                continue
            if edit_ratio(text, self.texts[index]) / 100.0 >= HELDOUT_EDIT:
                return index
        return None


def canonical_domain(record: dict[str, Any]) -> str:
    if record["language"] == "sw":
        return "unscripted"
    method = str(record.get("speech_method", "")).lower().replace("_", "-")
    if "unscript" in method or "spont" in method:
        return "unscripted"
    if "script" in method or "read" in method:
        return "scripted"
    role = str(record.get("adaptation_role", "")).lower()
    if role == "new_in_domain":
        return "unscripted"
    if role == "legacy_replay":
        return "scripted"
    raise AssertionError((record.get("sample_key"), method, role))


DEV_ALL = pd.read_parquet(DEV_SELECTION_PATH)
required_dev_columns = {
    "sample_key", "language", "split", "speech_method", "adaptation_role",
    "source_kind", "audio_ref", "parquet_path", "row_index", "text", "seconds",
}
assert required_dev_columns <= set(DEV_ALL.columns), required_dev_columns - set(DEV_ALL.columns)
assert DEV_ALL["sample_key"].astype(str).is_unique
assert DEV_ALL["split"].astype(str).eq("dev").all()
train_keys = set(
    pd.read_parquet(TRAIN_SELECTION_PATH, columns=["sample_key"])["sample_key"].astype(str)
)
assert set(DEV_ALL["sample_key"].astype(str)).isdisjoint(train_keys)
DEV_ALL["seconds"] = pd.to_numeric(DEV_ALL["seconds"], errors="raise")
assert DEV_ALL["seconds"].between(0.5, 38.0).all()
DEV_ALL["reference"] = DEV_ALL["text"].map(normalize_wer_text)
assert DEV_ALL["reference"].str.len().gt(0).all()
DEV_ALL["k2_text"] = DEV_ALL["text"].map(normalize_text)
DEV_ALL["decode_domain"] = [canonical_domain(row) for row in DEV_ALL.to_dict("records")]
DEV_ALL["group"] = DEV_ALL["language"].astype(str) + "|" + DEV_ALL["decode_domain"]
DEV_ALL = DEV_ALL[DEV_ALL["group"].isin(ACTIVE_GROUPS)].copy()

# Verification exacte contre le corpus K2 final. K2 garantit deja zero overlap
# exact ET near avec sa denylist dev/local_test ; cette verification independante
# empeche une regression de provenance du selected_dev.
K2_EXACT: dict[str, set[str]] = {}
K2_NEAR: dict[str, K2NearIndex] = {}
for language in LANGUAGES:
    path = K2_ROOT / f"final/{language}.parquet"
    assert path.is_file()
    table = pq.read_table(path, columns=["text_norm"])
    values = table.column("text_norm").to_pylist()
    K2_EXACT[language] = {
        hashlib.sha256(value.encode("utf-8")).hexdigest() for value in values
    }
    near_index = K2NearIndex()
    for value in values:
        near_index.add(value)
    K2_NEAR[language] = near_index

DEV_ALL["reference_sha256"] = DEV_ALL["reference"].map(
    lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()
)
DEV_ALL["train_exact_overlap"] = [
    any(
        hashlib.sha256(chunk.encode("utf-8")).hexdigest() in K2_EXACT[language]
        for chunk in chunk_k2_text(k2_text)
    )
    for language, k2_text in zip(DEV_ALL["language"], DEV_ALL["k2_text"])
]
DEV_ALL["train_near_overlap"] = [
    any(K2_NEAR[language].find(chunk) is not None for chunk in chunk_k2_text(k2_text))
    for language, k2_text in zip(DEV_ALL["language"], DEV_ALL["k2_text"])
]
leakage_counts = {
    "exact": int(DEV_ALL.train_exact_overlap.sum()),
    "near": int(DEV_ALL.train_near_overlap.sum()),
}
DEV_ALL = DEV_ALL[
    ~DEV_ALL.train_exact_overlap & ~DEV_ALL.train_near_overlap
].copy()
print("Filtre fuite K2 :", leakage_counts)


def deterministic_split(group: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    group = group.copy()
    before = len(group)
    # Une reference ne peut apparaitre qu'une fois dans le protocole.
    group["_rank"] = group.apply(
        lambda row: hashlib.sha256(
            f"20.K4|{row['group']}|{row['reference_sha256']}|{row['sample_key']}".encode("utf-8")
        ).hexdigest(),
        axis=1,
    )
    group = group.sort_values(["reference_sha256", "_rank"])
    group = group.drop_duplicates("reference_sha256", keep="first")
    group = group.sort_values("_rank").reset_index(drop=True)
    # Near-dedup interne avant le split, afin qu'une paraphrase quasi identique
    # ne puisse pas se retrouver de part et d'autre de tune/audit.
    internal_near = K2NearIndex()
    keep_indices = []
    for index, row in group.iterrows():
        chunks = chunk_k2_text(row["k2_text"])
        if any(internal_near.find(chunk) is not None for chunk in chunks):
            continue
        keep_indices.append(index)
        for chunk in chunks:
            internal_near.add(chunk)
    group = group.loc[keep_indices].reset_index(drop=True)
    assert len(group) >= AUDIT_MIN_CLIPS + 36, (group.group.iloc[0], len(group))
    audit_n = min(AUDIT_MAX_CLIPS, max(AUDIT_MIN_CLIPS, len(group) // 3))
    audit = group.iloc[:audit_n].copy()
    tune = group.iloc[audit_n : audit_n + TUNE_MAX_CLIPS].copy()
    assert len(tune) >= 36
    assert set(tune.reference_sha256).isdisjoint(set(audit.reference_sha256))
    metadata = {
        "before": before,
        "after_reference_dedup": len(group),
        "train_exact_removed": leakage_counts["exact"],
        "train_near_removed": leakage_counts["near"],
        "tune_clips": len(tune),
        "audit_clips": len(audit),
        "tune_words": int(tune.reference.map(lambda x: len(x.split())).sum()),
        "audit_words": int(audit.reference.map(lambda x: len(x.split())).sum()),
    }
    return tune.drop(columns="_rank"), audit.drop(columns="_rank"), metadata


GROUP_SPLITS: dict[str, dict[str, pd.DataFrame]] = {}
SPLIT_METADATA: dict[str, dict[str, Any]] = {}
split_records = []
for group_key in ACTIVE_GROUPS:
    group_frame = DEV_ALL[DEV_ALL.group.eq(group_key)].copy()
    tune, audit, metadata = deterministic_split(group_frame)
    GROUP_SPLITS[group_key] = {"tune": tune, "audit": audit}
    SPLIT_METADATA[group_key] = metadata
    for split_name, frame in (("tune", tune), ("audit", audit)):
        for row in frame.to_dict("records"):
            split_records.append(
                {
                    "group": group_key,
                    "split": split_name,
                    "sample_key_sha256": hashlib.sha256(
                        str(row["sample_key"]).encode("utf-8")
                    ).hexdigest(),
                    "reference_sha256": row["reference_sha256"],
                    "seconds": float(row["seconds"]),
                    "reference_words": len(row["reference"].split()),
                }
            )
SPLIT_FRAME = pd.DataFrame(split_records).sort_values(
    ["group", "split", "sample_key_sha256"]
).reset_index(drop=True)
SPLIT_SHA256 = canonical_sha256(SPLIT_FRAME.to_dict("records"))

print("\n=== SPLIT TUNE/AUDIT PRE-ENREGISTRE ===")
print(pd.DataFrame.from_dict(SPLIT_METADATA, orient="index").to_string())


# ---------------------------------------------------------------------------
# 5. Inventaire candidat et contrat de reprise
# ---------------------------------------------------------------------------

def resolve_unigrams(config: dict[str, Any], language: str) -> Path:
    candidates = []
    for key in (
        "uni_file", "unigrams", "unigram_file", "unigrams_file"
    ):
        if isinstance(config.get(key), (str, os.PathLike)) and config.get(key):
            candidates.append(Path(config[key]))
    binary = Path(config["lm_bin"])
    candidates.extend(
        [
            binary.with_name(f"{language}_unigrams.txt"),
            binary.with_name("unigrams.txt"),
        ]
    )
    path = next((item for item in candidates if item.is_file()), None)
    assert path is not None, (language, config)
    return path


LM_INVENTORY: dict[str, list[dict[str, Any]]] = {}
for group_key in ACTIVE_GROUPS:
    language, domain = group_key.split("|")
    baseline = BASELINE_CONFIG[group_key]
    baseline_binary = Path(baseline["lm_bin"])
    baseline_unigrams = resolve_unigrams(baseline, language)
    assets = [
        {
            "asset_id": "production",
            "family": "production_036878",
            "candidate": str(baseline.get("lm", baseline.get("lm_name", "production"))),
            "binary": str(baseline_binary),
            "binary_sha256": sha256_file(baseline_binary),
            "binary_bytes": baseline_binary.stat().st_size,
            "unigrams": str(baseline_unigrams),
            "unigrams_sha256": sha256_file(baseline_unigrams),
            "production_point": {
                "alpha": float(baseline["alpha"]),
                "beta": float(baseline["beta"]),
                "beam": int(baseline["beam"]),
            },
        }
    ]
    for finalist in K3_SELECTION[language]["v6_finalists"]:
        binary = K3_ROOT / finalist["binary_relative_path"]
        unigrams = K3_ROOT / finalist["unigrams_relative_path"]
        assets.append(
            {
                "asset_id": f"v6_{finalist['candidate']}",
                "family": "kenlm_v6_20_K3",
                "candidate": finalist["candidate"],
                "binary": str(binary),
                "binary_sha256": sha256_file(binary),
                "binary_bytes": binary.stat().st_size,
                "unigrams": str(unigrams),
                "unigrams_sha256": sha256_file(unigrams),
                "production_point": None,
            }
        )
    # Deduplication par contenu, pas par nom.
    unique = OrderedDict()
    for asset in assets:
        key = (asset["binary_sha256"], asset["unigrams_sha256"])
        if key not in unique or asset["asset_id"] == "production":
            unique[key] = asset
    LM_INVENTORY[group_key] = list(unique.values())
    assert any(item["asset_id"] == "production" for item in LM_INVENTORY[group_key])

ENVIRONMENT = {
    "created_at": now_iso(),
    "python": sys.version,
    "packages": package_versions(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": int(torch.cuda.get_device_properties(0).total_memory),
    "system_ram_bytes": int(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")),
}
OMNI_REPO = Path("/content/omnilingual-asr")
assert (OMNI_REPO / ".git").is_dir(), "Relancer la cellule 20.K4.A."
OMNI_REPO_COMMIT = subprocess.run(
    ["git", "-C", str(OMNI_REPO), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
SOFTWARE_FINGERPRINT = {
    "python": sys.version.split()[0],
    "packages": package_versions(),
    "omnilingual_asr_git_commit": OMNI_REPO_COMMIT,
}
SOFTWARE_FINGERPRINT_SHA256 = canonical_sha256(SOFTWARE_FINGERPRINT)

CONTRACT_PAYLOAD = {
    "schema": 1,
    "cell": "20.K4",
    "purpose": "leak-free acoustic WER tuning of K3 KenLM V6 finalists",
    "k2_run_id": EXPECTED_K2_RUN_ID,
    "k2_contract_sha256": EXPECTED_K2_CONTRACT_SHA256,
    "k3_run_id": EXPECTED_K3_RUN_ID,
    "k3_contract_sha256": EXPECTED_K3_CONTRACT_SHA256,
    "base_weight_sha256": BASE_WEIGHT_SHA256,
    "mas_adapter_sha256": MAS_ADAPTER_SHA256,
    "baseline_config_sha256": BASELINE_CONFIG_SHA256,
    "lora_plan_sha256": PLAN_SHA256,
    "dev_selection_sha256": DEV_SELECTION_SHA256,
    "train_selection_sha256": TRAIN_SELECTION_SHA256,
    "split_sha256": SPLIT_SHA256,
    "split_metadata": SPLIT_METADATA,
    "active_groups": list(ACTIVE_GROUPS),
    "lm_inventory": LM_INVENTORY,
    "software_fingerprint": SOFTWARE_FINGERPRINT,
    "software_fingerprint_sha256": SOFTWARE_FINGERPRINT_SHA256,
    "neural_parameters": TOTAL_NEURAL_PARAMETERS,
    "batch_size": BATCH_SIZE,
    "coarse_alphas": list(COARSE_ALPHAS),
    "coarse_betas": list(COARSE_BETAS),
    "fine_alpha_offsets": list(FINE_ALPHA_OFFSETS),
    "fine_beta_offsets": list(FINE_BETA_OFFSETS),
    "beam_widths_final": list(BEAM_WIDTHS_FINAL),
    "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
    "acceptance": {
        "minimum_audit_absolute_gain": MIN_AUDIT_ABSOLUTE_GAIN,
        "minimum_probability_better": MIN_PROBABILITY_BETTER,
        "maximum_duration_stratum_regression": MAX_DURATION_STRATUM_REGRESSION,
        "minimum_stratum_reference_words": MIN_STRATUM_REFERENCE_WORDS,
        "audit_is_never_used_for_candidate_selection": True,
        "second_candidate_not_tested_after_audit_failure": True,
    },
}
CONTRACT_SHA256 = canonical_sha256(CONTRACT_PAYLOAD)
RUN_ID = CONTRACT_SHA256[:16]
FINAL_RUN_DIR = K4_REPORT_ROOT / RUN_ID
STAGING_RUN_DIR = K4_REPORT_ROOT / f"{RUN_ID}.staging"
GROUP_DIR = STAGING_RUN_DIR / "groups"

CONTRACT_DOCUMENT = {
    "contract": CONTRACT_PAYLOAD,
    "contract_sha256": CONTRACT_SHA256,
}


def validate_completed_k4(path: Path) -> dict[str, Any] | None:
    try:
        complete = read_json(path / "_COMPLETE.json")
        assert complete["cell"] == "20.K4"
        assert complete["run_id"] == RUN_ID
        assert complete["contract_sha256"] == CONTRACT_SHA256
        assert complete["status"] in (
            "PASS_READY_FOR_20_K5_EDGE_AUDIT", "PASS_BASELINE_RETAINED"
        )
        required_artifacts = {
            "contract.json",
            "environment_fingerprint.json",
            "eval_split_manifest.parquet",
            "runtime_signature.json",
            "grid_summary.parquet",
            "grid_summary.csv",
            "decision_summary.csv",
            "kenlm_tuning_by_domain_v3_k4_candidate.json",
            "kenlm_acoustic_wer_20_K4.json",
            *{
                f"groups/{group_key.replace('|', '__')}.json"
                for group_key in ACTIVE_GROUPS
            },
        }
        assert set(complete["artifacts"]) == required_artifacts
        validate_artifacts(path, complete["artifacts"])
        assert read_json(path / "contract.json") == CONTRACT_DOCUMENT
        candidate = read_json(
            path / "kenlm_tuning_by_domain_v3_k4_candidate.json"
        )
        assert set(candidate) == set(ALL_RUNTIME_GROUPS)
        report = read_json(path / "kenlm_acoustic_wer_20_K4.json")
        assert report["run_id"] == RUN_ID
        assert report["contract_sha256"] == CONTRACT_SHA256
        assert report["status"] == complete["status"]
        assert report["candidate_config_sha256"] == sha256_file(
            path / "kenlm_tuning_by_domain_v3_k4_candidate.json"
        )
        for group_key in ACTIVE_GROUPS:
            group = read_json(
                path / "groups" / f"{group_key.replace('|', '__')}.json"
            )
            assert group["group"] == group_key
            assert group["contract_sha256"] == CONTRACT_SHA256
            assert group["status"] == "COMPLETE"
        return complete
    except Exception:
        return None


if FINAL_RUN_DIR.exists():
    existing_complete = validate_completed_k4(FINAL_RUN_DIR)
    assert existing_complete is not None, (
        "Un repertoire final 20.K4 existe mais est invalide", FINAL_RUN_DIR
    )
    final_report = FINAL_RUN_DIR / "kenlm_acoustic_wer_20_K4.json"
    final_config = FINAL_RUN_DIR / "kenlm_tuning_by_domain_v3_k4_candidate.json"
    latest_existing = {
        "schema": 1,
        "cell": "20.K4",
        "status": existing_complete["status"],
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA256,
        "report": str(final_report),
        "report_sha256": sha256_file(final_report),
        "complete": str(FINAL_RUN_DIR / "_COMPLETE.json"),
        "complete_sha256": sha256_file(FINAL_RUN_DIR / "_COMPLETE.json"),
        "candidate_config": str(final_config),
        "candidate_config_sha256": sha256_file(final_config),
        "active_runtime_modified": False,
        "submission_created": False,
    }
    atomic_json(latest_existing, K4_REPORT_ROOT / "LATEST_ATTEMPT.json")
    atomic_json(latest_existing, K4_REPORT_ROOT / "LATEST.json")
    atomic_json(latest_existing, K4_REPORT_ROOT / "LATEST_PASS.json")
    print("✅ 20.K4 deja termine et pointeurs repares :", FINAL_RUN_DIR)
    print("STATUT :", existing_complete["status"])
    raise SystemExit(0)

GROUP_DIR.mkdir(parents=True, exist_ok=True)
contract_path = STAGING_RUN_DIR / "contract.json"
if contract_path.exists():
    assert read_json(contract_path) == CONTRACT_DOCUMENT
else:
    atomic_json(CONTRACT_DOCUMENT, contract_path)
atomic_json(ENVIRONMENT, STAGING_RUN_DIR / "environment_fingerprint.json")
atomic_parquet(SPLIT_FRAME, STAGING_RUN_DIR / "eval_split_manifest.parquet")


# ---------------------------------------------------------------------------
# 6. Maternalisation audio locale et runtime acoustique exact
# ---------------------------------------------------------------------------

USED_FRAME = pd.concat(
    [
        GROUP_SPLITS[group_key][split_name]
        for group_key in ACTIVE_GROUPS
        for split_name in ("tune", "audit")
    ],
    ignore_index=True,
)


def audio_destination(row: dict[str, Any]) -> Path:
    digest = hashlib.sha256(
        f"{row['sample_key']}|{row['reference_sha256']}".encode("utf-8")
    ).hexdigest()[:24]
    return LOCAL_AUDIO_ROOT / DEV_SELECTION_SHA256[:12] / row["language"] / f"{digest}.flac"


USED_FRAME["eval_audio_path"] = [
    str(audio_destination(row)) for row in USED_FRAME.to_dict("records")
]
missing = USED_FRAME[~USED_FRAME.eval_audio_path.map(os.path.exists)].copy()
if not missing.empty:
    print("Maternalisation locale :", len(missing), "clips")
    for row in missing[missing.source_kind.eq("file")].to_dict("records"):
        source = Path(str(row["audio_ref"]))
        destination = audio_destination(row)
        assert source.is_file(), source
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
        shutil.copyfile(source, temporary)
        os.replace(temporary, destination)
    embedded = missing[~missing.source_kind.eq("file")]
    for parquet_path, frame in embedded.groupby("parquet_path", sort=True):
        parquet_path = Path(str(parquet_path))
        assert parquet_path.is_file(), parquet_path
        rows = pq.read_table(parquet_path, columns=["audio_bytes"]).to_pylist()
        for row in frame.to_dict("records"):
            destination = audio_destination(row)
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
            with open(temporary, "wb") as handle:
                handle.write(bytes(rows[int(row["row_index"])] ["audio_bytes"]))
            os.replace(temporary, destination)

assert USED_FRAME.eval_audio_path.map(os.path.exists).all()
audio_fingerprint_rows = []
for row in USED_FRAME.to_dict("records"):
    path = Path(row["eval_audio_path"])
    assert path.stat().st_size > 256, path
    info = sf.info(path)
    assert info.samplerate == 16_000 and info.frames > 0, (path, info)
    audio_fingerprint_rows.append(
        {
            "sample_key_sha256": hashlib.sha256(
                str(row["sample_key"]).encode("utf-8")
            ).hexdigest(),
            "audio_sha256": sha256_file(path),
            "bytes": path.stat().st_size,
            "frames": int(info.frames),
            "samplerate": int(info.samplerate),
        }
    )
AUDIO_FINGERPRINT_SHA256 = canonical_sha256(
    sorted(audio_fingerprint_rows, key=lambda row: row["sample_key_sha256"])
)
audio_map = dict(zip(USED_FRAME.sample_key.astype(str), USED_FRAME.eval_audio_path))
for group_key in ACTIVE_GROUPS:
    for split_name in ("tune", "audit"):
        frame = GROUP_SPLITS[group_key][split_name].copy()
        frame["eval_audio_path"] = frame.sample_key.astype(str).map(audio_map)
        assert frame.eval_audio_path.map(os.path.exists).all()
        GROUP_SPLITS[group_key][split_name] = frame


def write_edge_card() -> Path:
    # La carte doit exister AVANT le premier import d'omnilingual_asr : son
    # registre de cartes peut etre initialise au moment de l'import.
    import importlib.util

    spec = importlib.util.find_spec("omnilingual_asr")
    assert spec is not None and spec.submodule_search_locations
    package_root = Path(next(iter(spec.submodule_search_locations))).resolve()
    card_dir = package_root / "cards/models"
    card_dir.mkdir(parents=True, exist_ok=True)
    card_path = card_dir / f"{LOCAL_CARD_NAME}.yaml"
    payload = (
        f"name: {LOCAL_CARD_NAME}\n"
        "base: omniASR_CTC_1B_v2\n"
        f"checkpoint: \"file://{BASE_WEIGHT}\"\n"
    )
    if not card_path.exists() or card_path.read_text(encoding="utf-8") != payload:
        card_path.write_text(payload, encoding="utf-8")
    return card_path


EDGE_CARD_PATH = write_edge_card()

from fairseq2.data.tokenizers.hub import load_tokenizer
from fairseq2.models.hub import load_model
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline


def projection_dimensions(module: nn.Module) -> tuple[int, int] | None:
    weight = getattr(module, "weight", None)
    if not isinstance(weight, torch.Tensor) or weight.ndim != 2:
        return None
    name = type(module).__name__.lower()
    namespace = type(module).__module__.lower()
    if any(token in name for token in ("embedding", "embed", "conv", "norm")):
        return None
    if any(True for _ in module.children()):
        return None
    input_dim = getattr(module, "in_features", getattr(module, "input_dim", None))
    output_dim = getattr(module, "out_features", getattr(module, "output_dim", None))
    input_dim = int(input_dim) if input_dim is not None else int(weight.shape[1])
    output_dim = int(output_dim) if output_dim is not None else int(weight.shape[0])
    if tuple(weight.shape) != (output_dim, input_dim):
        return None
    if not (
        isinstance(module, nn.Linear)
        or "linear" in name
        or "projection" in name
        or "projection" in namespace
    ):
        return None
    return input_dim, output_dim


class EvalLoRAProjection(nn.Module):
    def __init__(self, base: nn.Module, rank: int):
        super().__init__()
        dimensions = projection_dimensions(base)
        assert dimensions is not None
        input_dim, output_dim = dimensions
        self.base = base
        self.rank = int(rank)
        self.scaling = 1.0
        self.enabled = False
        self.lora_A = nn.Parameter(
            torch.zeros(
                self.rank, input_dim, device=base.weight.device, dtype=base.weight.dtype
            ),
            requires_grad=False,
        )
        self.lora_B = nn.Parameter(
            torch.zeros(
                output_dim, self.rank, device=base.weight.device, dtype=base.weight.dtype
            ),
            requires_grad=False,
        )

    def forward(self, inputs: torch.Tensor, *args: Any, **kwargs: Any) -> torch.Tensor:
        output = self.base(inputs, *args, **kwargs)
        if not self.enabled:
            return output
        return output + F.linear(F.linear(inputs, self.lora_A), self.lora_B) * self.scaling


def parent_and_child(root: nn.Module, dotted_name: str) -> tuple[nn.Module, str]:
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


LORA_PLAN = pd.read_parquet(PLAN_PATH)
assert len(LORA_PLAN) == 289
assert int(LORA_PLAN["parameters_per_adapter"].sum()) == MAS_ADAPTER_PARAMETERS


def inject_eval_lora(root: nn.Module) -> OrderedDict[str, EvalLoRAProjection]:
    wrappers: OrderedDict[str, EvalLoRAProjection] = OrderedDict()
    for row in LORA_PLAN.sort_values("module").to_dict("records"):
        parent, child = parent_and_child(root, str(row["module"]))
        base = getattr(parent, child)
        assert projection_dimensions(base) == (
            int(row["in_features"]), int(row["out_features"])
        )
        wrapper = EvalLoRAProjection(base, int(row["rank"]))
        setattr(parent, child, wrapper)
        wrappers[str(row["module"])] = wrapper
    return wrappers


def labels_from_tokenizer(tokenizer: Any) -> list[str]:
    tokenizer_model = getattr(tokenizer, "_model", None)
    for method in ("index_to_token", "id_to_piece", "IdToPiece"):
        if tokenizer_model is not None and hasattr(tokenizer_model, method):
            labels = [
                str(getattr(tokenizer_model, method)(index))
                for index in range(tokenizer.vocab_info.size)
            ]
            labels[0] = ""
            assert len(labels) == tokenizer.vocab_info.size
            assert len(set(labels)) == len(labels), "Labels tokenizer non uniques."
            return labels
    raise RuntimeError("Vocabulaire tokenizer inaccessible")


for name in ("pipe", "model", "_pipe", "_pipe_lm", "_pipe_joint"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

print("Chargement du runtime BF16 exact 0.36878...")
model = load_model(LOCAL_CARD_NAME, device=torch.device("cuda"), dtype=torch.bfloat16)
assert sum(parameter.numel() for parameter in model.parameters()) == BASE_PARAMETERS
model.eval()
wrappers = inject_eval_lora(model)
assert len(wrappers) == 289
assert sum(parameter.numel() for parameter in model.parameters()) == (
    TOTAL_NEURAL_PARAMETERS
)

raw_adapter = torch.load(MAS_ADAPTER, map_location="cpu", weights_only=False)
assert raw_adapter.get("language") == "mas"
assert int(raw_adapter.get("step", -1)) == 1250
assert raw_adapter.get("plan_sha256") == PLAN_SHA256
adapter_state = raw_adapter.get("state_dict", raw_adapter)
assert isinstance(adapter_state, dict)
parameters = dict(model.named_parameters())
expected_adapter_keys = {
    f"{module_name}.lora_A" for module_name in wrappers
} | {
    f"{module_name}.lora_B" for module_name in wrappers
}
assert set(adapter_state) == expected_adapter_keys, (
    len(adapter_state), len(expected_adapter_keys), sorted(set(adapter_state) ^ expected_adapter_keys)[:5]
)
copied_adapter_parameters = 0
with torch.no_grad():
    for key, value in adapter_state.items():
        target = parameters[key]
        assert tuple(target.shape) == tuple(value.shape)
        target.copy_(value.to(device=target.device, dtype=target.dtype))
        copied_adapter_parameters += target.numel()
assert copied_adapter_parameters == MAS_ADAPTER_PARAMETERS
del raw_adapter, adapter_state, parameters

tokenizer = load_tokenizer(LOCAL_CARD_NAME)
LABELS = labels_from_tokenizer(tokenizer)
LABELS_SHA256 = hashlib.sha256("\0".join(LABELS).encode("utf-8")).hexdigest()
pipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)


def set_adapter_enabled(enabled: bool) -> None:
    for wrapper in wrappers.values():
        wrapper.enabled = bool(enabled)
    assert all(wrapper.enabled is bool(enabled) for wrapper in wrappers.values())


def capture_one(audio_path: str, omni_code: str) -> np.ndarray:
    captured: list[np.ndarray] = []
    original_forward = pipe.model.forward

    def spy(source_seqs: Any, source_seq_lens: Any, *args: Any, **kwargs: Any) -> Any:
        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)
        logits, layout = output[0], output[1]
        length = layout.seq_lens[0]
        length = int(length.item() if hasattr(length, "item") else length)
        value = torch.log_softmax(
            logits[0, :length].detach().float(), dim=-1
        ).cpu().numpy()
        captured.append(value)
        return output

    pipe.model.forward = spy
    try:
        with torch.inference_mode():
            pipe.transcribe([audio_path], lang=[omni_code], batch_size=1)
    finally:
        pipe.model.forward = original_forward
    assert len(captured) == 1
    value = captured[0]
    assert value.ndim == 2 and value.shape[1] == len(LABELS), value.shape
    assert np.isfinite(value).all()
    return value


# Le SHA labels depend du package charge et doit etre ajoute au contrat derive.
RUNTIME_SIGNATURE_PAYLOAD = {
    "labels_sha256": LABELS_SHA256,
    "labels": len(LABELS),
    "base_parameters": BASE_PARAMETERS,
    "mas_adapter_parameters": MAS_ADAPTER_PARAMETERS,
    "total_neural_parameters": TOTAL_NEURAL_PARAMETERS,
    "edge_card": str(EDGE_CARD_PATH),
    "edge_card_sha256": sha256_file(EDGE_CARD_PATH),
    "software_fingerprint_sha256": SOFTWARE_FINGERPRINT_SHA256,
    "used_audio_fingerprint_sha256": AUDIO_FINGERPRINT_SHA256,
    "used_audio_clips": len(audio_fingerprint_rows),
}
RUNTIME_SIGNATURE_SHA256 = canonical_sha256(RUNTIME_SIGNATURE_PAYLOAD)
RUNTIME_SIGNATURE = {
    "runtime": RUNTIME_SIGNATURE_PAYLOAD,
    "runtime_signature_sha256": RUNTIME_SIGNATURE_SHA256,
}
atomic_json(RUNTIME_SIGNATURE, STAGING_RUN_DIR / "runtime_signature.json")


# ---------------------------------------------------------------------------
# 7. Decodage, grille TUNE et audit apparie
# ---------------------------------------------------------------------------

def edit_counts(reference: str, hypothesis: str) -> dict[str, int]:
    result = jiwer.process_words(reference, hypothesis)
    return {
        "hits": int(result.hits),
        "substitutions": int(result.substitutions),
        "deletions": int(result.deletions),
        "insertions": int(result.insertions),
    }


def aggregate_counts(rows: Iterable[dict[str, int]]) -> dict[str, Any]:
    rows = list(rows)
    substitutions = sum(row["substitutions"] for row in rows)
    deletions = sum(row["deletions"] for row in rows)
    insertions = sum(row["insertions"] for row in rows)
    hits = sum(row["hits"] for row in rows)
    reference_words = hits + substitutions + deletions
    errors = substitutions + deletions + insertions
    assert reference_words > 0
    return {
        "hits": hits,
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "reference_words": reference_words,
        "errors": errors,
        "wer": errors / reference_words,
    }


def decode_hypotheses(
    decoder: Any,
    logits: list[np.ndarray],
    alpha: float,
    beta: float,
    beam: int,
) -> list[str]:
    assert hasattr(decoder, "reset_params"), "pyctcdecode.reset_params absent"
    decoder.reset_params(alpha=float(alpha), beta=float(beta))
    return [
        normalize_wer_text(decoder.decode(value, beam_width=int(beam)))
        for value in logits
    ]


def score_configuration(
    decoder: Any,
    logits: list[np.ndarray],
    references: list[str],
    alpha: float,
    beta: float,
    beam: int,
) -> tuple[float, list[dict[str, int]], list[str]]:
    hypotheses = decode_hypotheses(decoder, logits, alpha, beta, beam)
    counts = [edit_counts(ref, hyp) for ref, hyp in zip(references, hypotheses)]
    return float(aggregate_counts(counts)["wer"]), counts, hypotheses


def rounded_nonnegative(value: float) -> float:
    return round(max(0.0, float(value)), 4)


def localize_asset(asset: dict[str, Any]) -> tuple[Path, Path]:
    binary = copy_verified(
        Path(asset["binary"]), asset["binary_sha256"], f"{asset['asset_id']}_model"
    )
    unigrams = copy_verified(
        Path(asset["unigrams"]), asset["unigrams_sha256"], f"{asset['asset_id']}_unigrams"
    )
    return binary, unigrams


def build_decoder_for_asset(asset: dict[str, Any]) -> Any:
    binary, unigrams = localize_asset(asset)
    unigram_list = unigrams.read_text(encoding="utf-8").splitlines()
    assert unigram_list
    return build_ctcdecoder(
        LABELS,
        str(binary),
        unigrams=unigram_list,
        alpha=0.0,
        beta=0.0,
    )


def configuration_key(row: dict[str, Any]) -> tuple[Any, ...]:
    return (
        row["asset_id"],
        round(float(row["alpha"]), 4),
        round(float(row["beta"]), 4),
        int(row["beam"]),
    )


def paired_bootstrap(
    baseline_counts: list[dict[str, int]],
    challenger_counts: list[dict[str, int]],
    seed_offset: int,
) -> dict[str, float]:
    assert len(baseline_counts) == len(challenger_counts) and baseline_counts
    baseline_errors = np.asarray(
        [row["substitutions"] + row["deletions"] + row["insertions"] for row in baseline_counts],
        dtype=np.float64,
    )
    challenger_errors = np.asarray(
        [row["substitutions"] + row["deletions"] + row["insertions"] for row in challenger_counts],
        dtype=np.float64,
    )
    reference_words = np.asarray(
        [row["hits"] + row["substitutions"] + row["deletions"] for row in baseline_counts],
        dtype=np.float64,
    )
    assert np.all(reference_words > 0)
    rng = np.random.default_rng(BOOTSTRAP_SEED + seed_offset)
    deltas = np.empty(BOOTSTRAP_REPETITIONS, dtype=np.float64)
    n = len(reference_words)
    for index in range(BOOTSTRAP_REPETITIONS):
        sample = rng.integers(0, n, size=n)
        denominator = reference_words[sample].sum()
        deltas[index] = (
            challenger_errors[sample].sum() - baseline_errors[sample].sum()
        ) / denominator
    return {
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "probability_challenger_better": float(np.mean(deltas < 0.0)),
    }


def duration_bucket(seconds: float) -> str:
    if seconds <= 10:
        return "short_le_10s"
    if seconds <= 25:
        return "medium_10_25s"
    return "long_25_38s"


def duration_guard(
    audit_frame: pd.DataFrame,
    baseline_counts: list[dict[str, int]],
    challenger_counts: list[dict[str, int]],
) -> tuple[bool, list[dict[str, Any]]]:
    records = []
    buckets = [duration_bucket(float(value)) for value in audit_frame.seconds]
    for bucket in sorted(set(buckets)):
        indices = [index for index, value in enumerate(buckets) if value == bucket]
        baseline = aggregate_counts([baseline_counts[index] for index in indices])
        challenger = aggregate_counts([challenger_counts[index] for index in indices])
        delta = challenger["wer"] - baseline["wer"]
        eligible = baseline["reference_words"] >= MIN_STRATUM_REFERENCE_WORDS
        records.append(
            {
                "bucket": bucket,
                "clips": len(indices),
                "reference_words": baseline["reference_words"],
                "baseline_wer": baseline["wer"],
                "challenger_wer": challenger["wer"],
                "delta": delta,
                "guard_applies": eligible,
                "pass": (not eligible) or delta <= MAX_DURATION_STRATUM_REGRESSION,
            }
        )
    return all(record["pass"] for record in records), records


def group_result_path(group_key: str) -> Path:
    return GROUP_DIR / f"{group_key.replace('|', '__')}.json"


def valid_group_result(group_key: str) -> dict[str, Any] | None:
    path = group_result_path(group_key)
    if not path.is_file():
        return None
    try:
        value = read_json(path)
        assert value["contract_sha256"] == CONTRACT_SHA256
        assert value["group"] == group_key
        assert value["status"] == "COMPLETE"
        assert value["split_sha256"] == SPLIT_SHA256
        assert value["runtime_signature_sha256"] == RUNTIME_SIGNATURE_SHA256
        return value
    except Exception:
        return None


GROUP_RESULTS: dict[str, dict[str, Any]] = {}
GRID_ROWS: list[dict[str, Any]] = []

for group_index, group_key in enumerate(ACTIVE_GROUPS):
    resumed = valid_group_result(group_key)
    if resumed is not None:
        GROUP_RESULTS[group_key] = resumed
        GRID_ROWS.extend(resumed["grid_summary"])
        print("[deja termine]", group_key, "->", resumed["decision"])
        continue

    language, domain = group_key.split("|")
    tune_frame = GROUP_SPLITS[group_key]["tune"].reset_index(drop=True)
    audit_frame = GROUP_SPLITS[group_key]["audit"].reset_index(drop=True)
    combined = pd.concat([tune_frame, audit_frame], ignore_index=True)
    tune_n = len(tune_frame)

    print("\n" + "=" * 80)
    print(
        f"[{group_index + 1:02d}/{len(ACTIVE_GROUPS):02d}] {group_key}",
        "| tune", len(tune_frame), "| audit", len(audit_frame),
    )
    set_adapter_enabled(language == "mas")
    assert all(wrapper.enabled == (language == "mas") for wrapper in wrappers.values())
    started = time.monotonic()
    all_logits = []
    for row_index, row in enumerate(combined.to_dict("records"), 1):
        all_logits.append(capture_one(row["eval_audio_path"], LANG_TO_OMNI[language]))
        if row_index % 25 == 0 or row_index == len(combined):
            print(
                f"  logits {row_index}/{len(combined)} |",
                f"{(time.monotonic() - started) / 60:.1f} min",
            )
    tune_logits = all_logits[:tune_n]
    audit_logits = all_logits[tune_n:]
    tune_refs = tune_frame.reference.tolist()
    audit_refs = audit_frame.reference.tolist()
    del all_logits
    torch.cuda.empty_cache()

    group_grid: list[dict[str, Any]] = []
    per_asset_best: list[dict[str, Any]] = []
    baseline_tune_result = None
    baseline_asset = next(
        asset for asset in LM_INVENTORY[group_key] if asset["asset_id"] == "production"
    )

    for asset_index, asset in enumerate(LM_INVENTORY[group_key], 1):
        print(
            f"  LM {asset_index}/{len(LM_INVENTORY[group_key])} :",
            asset["asset_id"],
        )
        decoder = build_decoder_for_asset(asset)
        evaluated: dict[tuple[Any, ...], dict[str, Any]] = {}

        pairs = [(alpha, beta) for alpha in COARSE_ALPHAS for beta in COARSE_BETAS]
        if asset["production_point"] is not None:
            production = asset["production_point"]
            pairs.append((float(production["alpha"]), float(production["beta"])))
        for alpha, beta in pairs:
            row = {
                "asset_id": asset["asset_id"],
                "family": asset["family"],
                "candidate": asset["candidate"],
                "binary_sha256": asset["binary_sha256"],
                "unigrams_sha256": asset["unigrams_sha256"],
                "alpha": rounded_nonnegative(alpha),
                "beta": rounded_nonnegative(beta),
                "beam": 100,
                "phase": "coarse",
            }
            key = configuration_key(row)
            if key in evaluated:
                continue
            wer, _, _ = score_configuration(
                decoder, tune_logits, tune_refs, row["alpha"], row["beta"], row["beam"]
            )
            row["tune_wer"] = wer
            evaluated[key] = row

        coarse_best = min(
            evaluated.values(),
            key=lambda row: (row["tune_wer"], row["beam"], row["alpha"], row["beta"]),
        )
        for alpha_offset in FINE_ALPHA_OFFSETS:
            for beta_offset in FINE_BETA_OFFSETS:
                row = {
                    **{key: coarse_best[key] for key in (
                        "asset_id", "family", "candidate", "binary_sha256", "unigrams_sha256"
                    )},
                    "alpha": rounded_nonnegative(coarse_best["alpha"] + alpha_offset),
                    "beta": rounded_nonnegative(coarse_best["beta"] + beta_offset),
                    "beam": 100,
                    "phase": "fine",
                }
                key = configuration_key(row)
                if key in evaluated:
                    continue
                wer, _, _ = score_configuration(
                    decoder, tune_logits, tune_refs, row["alpha"], row["beta"], row["beam"]
                )
                row["tune_wer"] = wer
                evaluated[key] = row

        asset_best = min(
            evaluated.values(),
            key=lambda row: (row["tune_wer"], row["beam"], row["alpha"], row["beta"]),
        )
        per_asset_best.append(dict(asset_best))
        group_grid.extend(evaluated.values())

        if asset["asset_id"] == "production":
            point = asset["production_point"]
            baseline_row = {
                "asset_id": "production",
                "family": asset["family"],
                "candidate": asset["candidate"],
                "binary_sha256": asset["binary_sha256"],
                "unigrams_sha256": asset["unigrams_sha256"],
                "alpha": float(point["alpha"]),
                "beta": float(point["beta"]),
                "beam": int(point["beam"]),
                "phase": "production_exact",
            }
            key = configuration_key(baseline_row)
            if key in evaluated:
                baseline_tune_result = dict(evaluated[key])
            else:
                wer, _, _ = score_configuration(
                    decoder,
                    tune_logits,
                    tune_refs,
                    baseline_row["alpha"],
                    baseline_row["beta"],
                    baseline_row["beam"],
                )
                baseline_row["tune_wer"] = wer
                baseline_tune_result = baseline_row
                group_grid.append(baseline_row)
        del decoder
        gc.collect()

    assert baseline_tune_result is not None

    # Beam 250 uniquement pour les deux meilleurs actifs selon TUNE.
    top_two = sorted(
        per_asset_best,
        key=lambda row: (
            row["tune_wer"],
            0 if row["asset_id"] == "production" else 1,
            next(
                asset["binary_bytes"]
                for asset in LM_INVENTORY[group_key]
                if asset["asset_id"] == row["asset_id"]
            ),
        ),
    )[:2]
    for best in top_two:
        asset = next(
            asset for asset in LM_INVENTORY[group_key] if asset["asset_id"] == best["asset_id"]
        )
        decoder = build_decoder_for_asset(asset)
        row = {**best, "beam": 250, "phase": "beam250"}
        if not any(configuration_key(item) == configuration_key(row) for item in group_grid):
            wer, _, _ = score_configuration(
                decoder, tune_logits, tune_refs, row["alpha"], row["beta"], row["beam"]
            )
            row["tune_wer"] = wer
            group_grid.append(row)
        del decoder
        gc.collect()

    # Selection TUNE stricte. Une egalite a 1e-12 restaure la production.
    minimum_tune_wer = min(row["tune_wer"] for row in group_grid)
    tied = [row for row in group_grid if abs(row["tune_wer"] - minimum_tune_wer) <= 1e-12]
    tune_winner = min(
        tied,
        key=lambda row: (
            0 if configuration_key(row) == configuration_key(baseline_tune_result) else 1,
            next(
                asset["binary_bytes"]
                for asset in LM_INVENTORY[group_key]
                if asset["asset_id"] == row["asset_id"]
            ),
            row["beam"],
            row["alpha"],
            row["beta"],
        ),
    )

    # AUDIT : exactement deux configurations, production et gagnant TUNE.
    production_decoder = build_decoder_for_asset(baseline_asset)
    baseline_audit_wer, baseline_audit_counts, _ = score_configuration(
        production_decoder,
        audit_logits,
        audit_refs,
        baseline_tune_result["alpha"],
        baseline_tune_result["beta"],
        baseline_tune_result["beam"],
    )
    del production_decoder
    winner_asset = next(
        asset for asset in LM_INVENTORY[group_key] if asset["asset_id"] == tune_winner["asset_id"]
    )
    winner_decoder = build_decoder_for_asset(winner_asset)
    winner_audit_wer, winner_audit_counts, _ = score_configuration(
        winner_decoder,
        audit_logits,
        audit_refs,
        tune_winner["alpha"],
        tune_winner["beta"],
        tune_winner["beam"],
    )
    del winner_decoder
    gc.collect()

    audit_delta = winner_audit_wer - baseline_audit_wer
    bootstrap = paired_bootstrap(
        baseline_audit_counts, winner_audit_counts, seed_offset=group_index * 10_000
    )
    duration_pass, duration_rows = duration_guard(
        audit_frame, baseline_audit_counts, winner_audit_counts
    )
    winner_is_exact_baseline = (
        configuration_key(tune_winner) == configuration_key(baseline_tune_result)
    )
    accepted = (
        (not winner_is_exact_baseline)
        and audit_delta <= -MIN_AUDIT_ABSOLUTE_GAIN
        and bootstrap["probability_challenger_better"] >= MIN_PROBABILITY_BETTER
        and duration_pass
    )
    deployed = tune_winner if accepted else baseline_tune_result
    deployed_asset = next(
        asset for asset in LM_INVENTORY[group_key] if asset["asset_id"] == deployed["asset_id"]
    )

    if not accepted:
        # Restauration byte-semantique de la configuration ayant produit 0.36878.
        selected_config = json.loads(json.dumps(BASELINE_CONFIG[group_key]))
    else:
        # Ne pas faire heriter au V6 les anciens WER/signatures de 18.8.
        stale_prefixes = (
            "wer", "n_tune", "n_holdout", "selection", "decision",
            "tuning", "audit", "delta", "probability", "score",
        )
        selected_config = {
            key: value
            for key, value in BASELINE_CONFIG[group_key].items()
            if not any(str(key).lower().startswith(prefix) for prefix in stale_prefixes)
        }
        selected_config.update(
            {
                "lm": deployed_asset["candidate"],
                "lm_bin": deployed_asset["binary"],
                "uni_file": deployed_asset["unigrams"],
                "alpha": float(deployed["alpha"]),
                "beta": float(deployed["beta"]),
                "beam": int(deployed["beam"]),
            }
        )

    sample_audit = []
    for index, row in audit_frame.iterrows():
        sample_audit.append(
            {
                "sample_key_sha256": hashlib.sha256(
                    str(row["sample_key"]).encode("utf-8")
                ).hexdigest(),
                "reference_sha256": row["reference_sha256"],
                "seconds": float(row["seconds"]),
                "reference_words": len(row["reference"].split()),
                "baseline": baseline_audit_counts[index],
                "challenger": winner_audit_counts[index],
            }
        )

    for row in group_grid:
        row.update({"group": group_key, "language": language, "domain": domain})
    result = {
        "schema": 1,
        "cell": "20.K4",
        "status": "COMPLETE",
        "finished_at": now_iso(),
        "contract_sha256": CONTRACT_SHA256,
        "split_sha256": SPLIT_SHA256,
        "runtime_signature_sha256": RUNTIME_SIGNATURE_SHA256,
        "group": group_key,
        "language": language,
        "domain": domain,
        "tune_clips": len(tune_frame),
        "audit_clips": len(audit_frame),
        "baseline_tune": baseline_tune_result,
        "tune_winner": tune_winner,
        "audit": {
            "baseline_wer": baseline_audit_wer,
            "challenger_wer": winner_audit_wer,
            "delta": audit_delta,
            **bootstrap,
            "duration_strata": duration_rows,
        },
        "accepted": accepted,
        "decision": "accept_tune_winner" if accepted else "restore_production_036878",
        "selected_config": selected_config,
        "grid_summary": group_grid,
        "sample_audit_counts": sample_audit,
        "raw_references_or_hypotheses_written": False,
    }
    atomic_json(result, group_result_path(group_key))
    GROUP_RESULTS[group_key] = result
    GRID_ROWS.extend(group_grid)
    print(
        "✅", group_key,
        "| tune winner", tune_winner["asset_id"],
        "| audit delta", f"{audit_delta:+.5f}",
        "| P", f"{bootstrap['probability_challenger_better']:.3f}",
        "|", result["decision"],
    )
    del tune_logits, audit_logits
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# 8. Publication candidate uniquement, jamais activation
# ---------------------------------------------------------------------------

assert set(GROUP_RESULTS) == set(ACTIVE_GROUPS)
FINAL_CONFIG = json.loads(json.dumps(BASELINE_CONFIG))
for group_key, result in GROUP_RESULTS.items():
    FINAL_CONFIG[group_key] = result["selected_config"]
assert set(FINAL_CONFIG) == set(ALL_RUNTIME_GROUPS)
assert FINAL_CONFIG["sw|scripted"] == BASELINE_CONFIG["sw|scripted"]

accepted_groups = [
    group_key for group_key, result in GROUP_RESULTS.items() if result["accepted"]
]
status = (
    "PASS_READY_FOR_20_K5_EDGE_AUDIT"
    if accepted_groups
    else "PASS_BASELINE_RETAINED"
)

grid_frame = pd.DataFrame(GRID_ROWS).sort_values(
    ["group", "tune_wer", "asset_id", "beam", "alpha", "beta"]
)
atomic_parquet(grid_frame, STAGING_RUN_DIR / "grid_summary.parquet")
atomic_csv(grid_frame, STAGING_RUN_DIR / "grid_summary.csv")
candidate_config_path = (
    STAGING_RUN_DIR / "kenlm_tuning_by_domain_v3_k4_candidate.json"
)
atomic_json(FINAL_CONFIG, candidate_config_path)

summary_rows = []
for group_key in ACTIVE_GROUPS:
    result = GROUP_RESULTS[group_key]
    summary_rows.append(
        {
            "group": group_key,
            "accepted": result["accepted"],
            "decision": result["decision"],
            "winner_asset": result["tune_winner"]["asset_id"],
            "tune_winner_wer": result["tune_winner"]["tune_wer"],
            "baseline_tune_wer": result["baseline_tune"]["tune_wer"],
            "baseline_audit_wer": result["audit"]["baseline_wer"],
            "challenger_audit_wer": result["audit"]["challenger_wer"],
            "audit_delta": result["audit"]["delta"],
            "probability_better": result["audit"]["probability_challenger_better"],
        }
    )
summary_frame = pd.DataFrame(summary_rows)
atomic_csv(summary_frame, STAGING_RUN_DIR / "decision_summary.csv")

REPORT = {
    "schema": 1,
    "cell": "20.K4",
    "status": status,
    "finished_at": now_iso(),
    "run_id": RUN_ID,
    "contract_sha256": CONTRACT_SHA256,
    "k2_run_id": EXPECTED_K2_RUN_ID,
    "k3_run_id": EXPECTED_K3_RUN_ID,
    "runtime_baseline": "leaderboard_0.36878_step1250_bf16_plus_lora_mas",
    "neural_parameters": TOTAL_NEURAL_PARAMETERS,
    "active_groups": list(ACTIVE_GROUPS),
    "accepted_groups": accepted_groups,
    "groups": {
        key: {
            "decision": value["decision"],
            "accepted": value["accepted"],
            "audit": value["audit"],
            "selected_config": value["selected_config"],
        }
        for key, value in GROUP_RESULTS.items()
    },
    "candidate_config": str(
        FINAL_RUN_DIR / "kenlm_tuning_by_domain_v3_k4_candidate.json"
    ),
    "candidate_config_sha256": sha256_file(candidate_config_path),
    "baseline_config_sha256": BASELINE_CONFIG_SHA256,
    "active_runtime_modified": False,
    "submission_created": False,
    "test_audio_read": False,
    "raw_references_or_hypotheses_written": False,
    "next_step": (
        "20.K5 — audit edge/RSS de la configuration candidate puis seulement ensuite inference test"
        if accepted_groups
        else "Conserver le runtime 0.36878 ; aucun candidat V6 n'a franchi l'audit"
    ),
}
atomic_json(REPORT, STAGING_RUN_DIR / "kenlm_acoustic_wer_20_K4.json")

artifact_relatives = [
    "contract.json",
    "environment_fingerprint.json",
    "eval_split_manifest.parquet",
    "runtime_signature.json",
    "grid_summary.parquet",
    "grid_summary.csv",
    "decision_summary.csv",
    "kenlm_tuning_by_domain_v3_k4_candidate.json",
    "kenlm_acoustic_wer_20_K4.json",
    *[
        f"groups/{group_key.replace('|', '__')}.json"
        for group_key in ACTIVE_GROUPS
    ],
]
ARTIFACTS = {
    relative: artifact_metadata(STAGING_RUN_DIR / relative)
    for relative in artifact_relatives
}
COMPLETE = {
    "schema": 1,
    "cell": "20.K4",
    "status": status,
    "finished_at": now_iso(),
    "run_id": RUN_ID,
    "contract_sha256": CONTRACT_SHA256,
    "accepted_groups": accepted_groups,
    "artifacts": ARTIFACTS,
    "active_runtime_modified": False,
    "submission_created": False,
    "test_audio_read": False,
}
atomic_json(COMPLETE, STAGING_RUN_DIR / "_COMPLETE.json")
assert validate_completed_k4(STAGING_RUN_DIR) is not None
assert not FINAL_RUN_DIR.exists()
os.replace(STAGING_RUN_DIR, FINAL_RUN_DIR)
assert validate_completed_k4(FINAL_RUN_DIR) is not None

latest = {
    "schema": 1,
    "cell": "20.K4",
    "status": status,
    "run_id": RUN_ID,
    "contract_sha256": CONTRACT_SHA256,
    "report": str(FINAL_RUN_DIR / "kenlm_acoustic_wer_20_K4.json"),
    "report_sha256": sha256_file(FINAL_RUN_DIR / "kenlm_acoustic_wer_20_K4.json"),
    "complete": str(FINAL_RUN_DIR / "_COMPLETE.json"),
    "complete_sha256": sha256_file(FINAL_RUN_DIR / "_COMPLETE.json"),
    "candidate_config": str(
        FINAL_RUN_DIR / "kenlm_tuning_by_domain_v3_k4_candidate.json"
    ),
    "candidate_config_sha256": sha256_file(
        FINAL_RUN_DIR / "kenlm_tuning_by_domain_v3_k4_candidate.json"
    ),
    "active_runtime_modified": False,
    "submission_created": False,
}
atomic_json(latest, K4_REPORT_ROOT / "LATEST_ATTEMPT.json")
atomic_json(latest, K4_REPORT_ROOT / "LATEST.json")
atomic_json(latest, K4_REPORT_ROOT / "LATEST_PASS.json")

print("\n=== DECISIONS 20.K4 ===")
print(summary_frame.to_string(index=False))
print("\nSTATUT 20.K4 :", status)
print("Groupes V6 acceptes :", accepted_groups or "aucun")
print("Configuration candidate :", latest["candidate_config"])
print("Rapport :", latest["report"])
print("Aucun runtime actif et aucune soumission n'ont ete modifies.")


## 20.K4.C — Résultat publié (lecture seule)


In [ ]:
# 20.K4.C — Lecture seule du résultat publié
import json
from pathlib import Path
import pandas as pd
root = Path("/content/afrivoices_project/finetune_runs/experiment_run/reports/kenlm_v6_20_K4")
latest_path = root / "LATEST_PASS.json"
assert latest_path.is_file(), "20.K4.B n'est pas encore terminé."
latest = json.load(open(latest_path, encoding="utf-8"))
print(json.dumps(latest, indent=2, ensure_ascii=False))
summary = Path(latest["report"]).parent / "decision_summary.csv"
display(pd.read_csv(summary))
print("\nRappel : ne pas remplacer la configuration active avant l'audit edge 20.K5.")
